In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REUSE_CURRENT = True
REUSE_LEGACY = True
OUT_ROOT = "/content/drive/MyDrive/OpenPlaque/GPU_Coronary_Artery_Ensemble_v1"
SAVE_PROBABILITIES = True


# OpenPlaque — GPU coronary-artery segmentation ensemble

Runs the current TotalSegmentator `coronary_arteries` model and the previous `coronary_arteries_LEGACY` model on the frozen source CCTA. The two masks are preserved separately; intersection, union, and disagreement masks quantify model uncertainty before artery-specific extraction.

This is an independent segmentation experiment, not a replacement for validated centerlines. Research use only.

In [ ]:
!pip -q install TotalSegmentator SimpleITK nibabel pandas matplotlib
import os, sys, json, shutil, subprocess, zipfile
from pathlib import Path
import numpy as np, pandas as pd, SimpleITK as sitk, matplotlib.pyplot as plt

repo=Path("/content/OpenPlaque")
if repo.exists(): shutil.rmtree(repo)
!git clone -q --depth 1 --branch coronary-lumen-segmentation-from-main https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!nvidia-smi
!TotalSegmentator --version
!totalseg_info --classes -ta coronary_arteries


In [ ]:
# Reconstruct a geometry-correct source CCTA NIfTI from the frozen cache.
cache = Path("/content/drive/MyDrive/OpenPlaque/Cache/Secondary_3D_Vesselness_Topology_v1")
arr_path = cache/"series7_int16.npy"
meta_path = cache/"series7_int16.json"
if not arr_path.exists() or not meta_path.exists():
    raise FileNotFoundError("Missing frozen source-CCTA cache")

meta = json.loads(meta_path.read_text())
arr = np.load(arr_path, mmap_mode="r")
img = sitk.GetImageFromArray(arr)
spacing_zyx = np.asarray(meta["spacing_zyx"], float)
img.SetSpacing(tuple(spacing_zyx[::-1]))
origin = tuple(np.asarray(meta["positions_lps_mm"][0], float))
img.SetOrigin(origin)

iop = np.asarray(meta["image_orientation_patient"], float)
row = iop[:3]; col = iop[3:]
slc = np.cross(row, col)
direction = np.array([[row[0], col[0], slc[0]],
                      [row[1], col[1], slc[1]],
                      [row[2], col[2], slc[2]]], float)
img.SetDirection(tuple(direction.ravel()))
source_nii = Path("/content/source_ccta.nii.gz")
sitk.WriteImage(img, str(source_nii))
print(source_nii, arr.shape, img.GetSpacing(), img.GetOrigin(), img.GetDirection())


In [ ]:
out_root=Path(OUT_ROOT); out_root.mkdir(parents=True, exist_ok=True)

def run_totalseg(task, name, reuse):
    out = out_root/name
    mask = out/"coronary_arteries.nii.gz"
    if reuse and mask.exists():
        print("reuse", task, mask)
        return mask
    if out.exists(): shutil.rmtree(out)
    out.mkdir(parents=True)
    cmd=["TotalSegmentator","-i",str(source_nii),"-o",str(out),"-ta",task,"--device","gpu"]
    if SAVE_PROBABILITIES:
        cmd += ["--save_probabilities", str(out/f"{name}_probabilities.npz")]
    print("RUN:", " ".join(cmd))
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError:
        if SAVE_PROBABILITIES:
            print("Probability output failed; retrying mask-only.")
            shutil.rmtree(out); out.mkdir(parents=True)
            cmd=["TotalSegmentator","-i",str(source_nii),"-o",str(out),"-ta",task,"--device","gpu"]
            subprocess.run(cmd, check=True)
        else:
            raise
    if not mask.exists():
        found=list(out.rglob("*coronary*.nii.gz"))
        if len(found)!=1: raise RuntimeError(f"Cannot identify mask for {task}: {found}")
        shutil.copy2(found[0],mask)
    return mask

current = run_totalseg("coronary_arteries","current",REUSE_CURRENT)
legacy = run_totalseg("coronary_arteries_LEGACY","legacy",REUSE_LEGACY)


In [ ]:
# Consensus/disagreement in source geometry.
ci=sitk.ReadImage(str(current)); li=sitk.ReadImage(str(legacy))
ca=sitk.GetArrayFromImage(ci)>0; la=sitk.GetArrayFromImage(li)>0
if ca.shape != la.shape:
    raise RuntimeError((ca.shape,la.shape))

intersection = ca & la
union = ca | la
disagreement = ca ^ la
for name, a in [("intersection",intersection),("union",union),("disagreement",disagreement)]:
    x=sitk.GetImageFromArray(a.astype(np.uint8)); x.CopyInformation(ci)
    sitk.WriteImage(x,str(out_root/f"coronary_{name}.nii.gz"))

voxel_mm3=float(np.prod(ci.GetSpacing()))
stats=pd.DataFrame([
    {"mask":"current","voxels":int(ca.sum()),"volume_mm3":float(ca.sum()*voxel_mm3)},
    {"mask":"legacy","voxels":int(la.sum()),"volume_mm3":float(la.sum()*voxel_mm3)},
    {"mask":"intersection","voxels":int(intersection.sum()),"volume_mm3":float(intersection.sum()*voxel_mm3)},
    {"mask":"union","voxels":int(union.sum()),"volume_mm3":float(union.sum()*voxel_mm3)},
    {"mask":"disagreement","voxels":int(disagreement.sum()),"volume_mm3":float(disagreement.sum()*voxel_mm3)},
])
dice=2*intersection.sum()/max(ca.sum()+la.sum(),1)
stats["current_vs_legacy_dice"]=dice
stats.to_csv(out_root/"coronary_ensemble_summary.csv",index=False)
display(stats)


In [ ]:
# Automatically choose the axial slices with the most coronary mask for QC.
src=sitk.GetArrayFromImage(sitk.ReadImage(str(source_nii)))
score=union.sum(axis=(1,2))
ids=np.argsort(score)[-6:][::-1]
fig,axes=plt.subplots(2,3,figsize=(14,9))
for ax,z in zip(axes.ravel(),ids):
    ax.imshow(src[z],cmap="gray",vmin=-200,vmax=900)
    ax.imshow(intersection[z],alpha=.45)
    ax.imshow(disagreement[z],alpha=.55)
    ax.set_title(f"z={z}: intersection + disagreement")
    ax.axis("off")
fig.tight_layout(); fig.savefig(out_root/"coronary_ensemble_qc.png",dpi=180); plt.show()


In [ ]:
# Compact report. Full masks/probabilities remain in Drive.
summary={"status":"COMPLETE","models":["coronary_arteries","coronary_arteries_LEGACY"],
         "dice_current_vs_legacy":float(dice),
         "interpretation":"Intersection is high-confidence model agreement; union is sensitive extent; XOR is model disagreement. Artery-specific lumen metrics will be extracted against frozen RCA/LAD centerlines later."}
(out_root/"summary.json").write_text(json.dumps(summary,indent=2))
html=out_root/"OPENPLAQUE_GPU_CORONARY_ARTERY_ENSEMBLE_REPORT.html"
html.write_text("<html><body><h1>OpenPlaque GPU coronary-artery ensemble</h1><p>Research use only.</p>"+
                stats.to_html(index=False)+"<p><img src='coronary_ensemble_qc.png' style='max-width:100%'></p></body></html>")
zip_path=out_root/"OPENPLAQUE_GPU_CORONARY_ARTERY_ENSEMBLE_REPORT_BACK.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in [html,out_root/"summary.json",out_root/"coronary_ensemble_summary.csv",out_root/"coronary_ensemble_qc.png",
              out_root/"coronary_intersection.nii.gz",out_root/"coronary_union.nii.gz",out_root/"coronary_disagreement.nii.gz"]:
        z.write(p,p.name)
print("FINAL:",zip_path)
